# IRIS FUV Spectrograph Background Subtraction
This notebook is meant to remove any background, indiscriminately, from around the Si IV doublet lines found in the IRIS FUV spectrograph images. Known existing lines around the Si IV doublet are also preserved.

This notebook is applied to "level 1.1" dark-subtracted, fixed-pattern-removed, despiked spectrograph images, which come from passing level 1 data through
`apply_dark_sub_iris_prep.pro`, `removed_fixed_pattern.ipynb`, and then `despike_and_save.ipynb`. This notebook also requires the rolling trimmed mean array output from `apply_rolling_trimmed_mean.ipynb` for step 1 of the background subtraction. This notebook creates "level 1.2" data.

#### Import Statements

In [ ]:
%reload_ext autoreload
%autoreload 2
# %matplotlib notebook
%matplotlib inline

import pathlib as pl
import numpy as np
import pickle
import numba
import matplotlib.pyplot as plt
# params = {"ytick.color" : "k",
#           "xtick.color" : "k",
#           "axes.labelcolor" : "k",
#           "axes.edgecolor" : "k"}
# plt.rcParams.update(params)
from mpl_toolkits.axes_grid1 import ImageGrid
# import pandas as pd
from matplotlib import colors
import astropy.units as u
from astropy import constants as const
from iris_mosaics import wcs_to_bins, spectral_plot, read_sg_image, read_sg_image_lvl1
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support, time_support
from scipy.optimize import curve_fit, minimize
from astropy.modeling import models, fitting
from astropy.convolution import convolve, Gaussian1DKernel, Gaussian2DKernel
import astropy.time
import pyinterp.fill
import regridding
from scipy import interpolate
from scipy.signal import correlate2d
import scipy.ndimage
# import cv2
from astropy.io import fits
from astropy.wcs import WCS
from scipy.stats import trim_mean, mode
# from photutils.background import MeanBackground, MedianBackground, ModeEstimatorBackground
quantity_support()
time_support()

## Load data

#### File paths of Level 1.1 FDM data and FUV background image

In [ ]:
# path = pl.Path(iris_fdm.__file__).parent / 'data' / 'unstitched_mosaic'
path = pl.Path(r'D:\IRIS data\deep_mosaics\20240811')

# Path of Level 1.1 spectrograph image files (used to save the files at the end using their original names but in a new folder)
path_fdm = path / 'level_11_iris_prep_bgsub_fixed_pattern_removed'
files = list(path_fdm.glob('*.fits'))

# Paths of pickled despiked Level 1.1 data
path_dspk_1394 = path / 'level_11_fpr_despiked_1394.pickle'
path_dspk_1403 = path / 'level_11_fpr_despiked_1403.pickle'

# Paths of pickled rolling trimmed mean Level 1.1 data
path_rtm_1394 = path / 'level_11_fpr_despiked_rtm_1394.pickle'
path_rtm_1403 = path / 'level_11_fpr_despiked_rtm_1403.pickle'

# Path of FUV background image
path_bg = path.parent / 'fuv_background' / '20200301_222038_FUV_background.fits'

#### Select Si IV 1394 & 1403 regions
Manually choose area where data is -- WCS doesn't seem to work -- my guess is because the geometric correction hasn't been applied yet.

In [ ]:
# Number of images
num_imgs = len(files)

# Read in an image in order to see where to crop it down:
file_num = 1000
w_0, hdu_0, _ = read_sg_image(files[file_num],'fuv2')
img_0 = hdu_0[0].data

# Lengths of dimensions of total image
num_y_total = img_0.shape[0]
num_x_total = img_0.shape[1]

# Create masks for the areas of interest (Si IV 1394 & 1403).
# sl_1394 = slice(10,~9), slice(732,~215)
# sl_1403 = slice(10,~9), slice(868,~7)

# For the Aug 2024 mosaic that is twice as wide in the x direction...
x1 = 2 * 732
x2 = 2 * 215
x3 = 2 * 868
x4 = 2 * 8
sl_1394 = slice(10,~9), slice(x1,~x2)
sl_1403 = slice(10,~9), slice(x3,~x4)

# Si IV 1394 and 1403 image dimension lengths
num_y = img_0[sl_1394].shape[0]
num_x_1394 = img_0[sl_1394].shape[1]
num_x_1403 = img_0[sl_1403].shape[1]

# Treat the upper and lower parts of the CCD separately
# There is an interface where the two detector taps sit next to each other, causing a discontinuity in the image.

# Define the 1394 upper and lower slices
sl_1394_up = slice(0,264)
sl_1394_down = slice(264, None)

# Define the 1403 upper and lower slices
sl_1403_up = slice(0,264)
sl_1403_down = slice(264, None)

# New y-axis shape
num_y_split = img_0[sl_1394_up].shape[0]

# # Inspect resulting images (may need to enlarge to see whether the edges are NaNs)
# plt.figure(figsize=(5,15))
# plt.imshow(img_0[sl_1394], vmin=np.percentile(img_0[sl_1394],0), vmax=np.percentile(img_0[sl_1394], 99))
#
# plt.figure(figsize=(5,15))
# plt.imshow(img_0[sl_1403], vmin=np.percentile(img_0[sl_1403],0), vmax=np.percentile(img_0[sl_1403], 99))

#### Read in despiked Level 1.1 data

In [ ]:
with open(str(path_dspk_1394), 'rb') as f:
    sg_1394_dspk = pickle.load(f)

with open(str(path_dspk_1403), 'rb') as f:
    sg_1403_dspk = pickle.load(f)
    
# Create NaN mask
nan_mask_1394 = ~np.isfinite(sg_1394_dspk)
nan_mask_1403 = ~np.isfinite(sg_1403_dspk)

# Mean images of the despiked data
sg_1394_dspk_mean_image = np.nanmean(sg_1394_dspk, axis=0)
sg_1403_dspk_mean_image = np.nanmean(sg_1403_dspk, axis=0)

#### Read in rolling trimmed mean arrays

In [ ]:
with open(str(path_rtm_1394), 'rb') as f:
    average_1394 = pickle.load(f)

with open(str(path_rtm_1403), 'rb') as f:
    average_1403 = pickle.load(f)

# Mean of rolling trimmed mean arrays
average_1394_mean_image = np.nanmean(average_1394, axis=0)
average_1403_mean_image = np.nanmean(average_1403, axis=0)

#### Read in FUV background image

In [ ]:
# # Read in normalized FUV background image data
# wcs_bg, hdul_bg, _ = read_sg_image(path_bg)
# fuv_bg = hdul_bg[0].data
#
# # Rebin FUV background to match science FDM spectrograph image dimensions
# fuv_bg = fuv_bg.reshape((fuv_bg.shape[0]//2, 2, fuv_bg.shape[1]//4, 4))
# fuv_bg = fuv_bg.mean((1,3))
#
# # Define FUV2 section of background image
# bg_1394 = fuv_bg[sl_1394]
# bg_1403 = fuv_bg[sl_1403]

#### Define plot function to better display Si IV 1394 and 1403 together

In [ ]:
def plot_lines_sidebyside(
        si_iv_1394, si_iv_1403, title,
        percentile_min: float = 0,
        percentile_max: float = 100,
        size: tuple = (5,5),
        exp_min = None,
        exp_max = None,
):
    # Set up figure and image grid
    fig = plt.figure(figsize=size)
    grid = ImageGrid(fig,
                     111,          # as in plt.subplot(111)
                     nrows_ncols=(1,2),
                     axes_pad=0.15,
                     # share_all=True,
                     cbar_location="right",
                     cbar_mode="single",
                     cbar_size="20%",
                     cbar_pad=0.15,
                     )

    if exp_min is None:
        exp_min = np.nanpercentile(si_iv_1394, percentile_min)
    if exp_max is None:
        exp_max = np.nanpercentile(si_iv_1394, percentile_max)

    im1 = grid[0].imshow(si_iv_1394, vmin=exp_min, vmax=exp_max)
    im2 = grid[1].imshow(si_iv_1403, vmin=exp_min, vmax=exp_max)
    # grid[0].set_title('1394 $\AA$', color='white')
    # grid[1].set_title('1403 $\AA$', color='white')
    grid[0].set_title('1394 Å')
    grid[1].set_title('1403 Å')
    # fig.suptitle(title, color='white')
    fig.suptitle(title)

    # Colorbar
    grid[~0].cax.colorbar(im2).set_label('DN', rotation=270)
    # grid[~0].cax.toggle_label(True)
    grid[0].invert_yaxis()
    # plt.tight_layout()    # Works, but may still require rect parameter to keep colorbar labels visible
    # plt.show()
    return fig

In [ ]:
plot_lines_sidebyside(
    sg_1394_dspk_mean_image,
    sg_1403_dspk_mean_image,
    '',
    # percentile_min=.1,
    # percentile_max=100,
    exp_max=40,
    exp_min=-3,
    size=(7,10),
);
# .savefig('despiked_mean_may_2025.png',dpi=300, transparent=True)

## Create own background images

The existing background subtraction method in `iris_prep.pro` does not remove enough of the higher-order background in the images, so we have another approach to create background images

The background subtraction is handled in two steps in order to treat the two kinds of background features we've encountered:
1. Features that change slowly in time, and quickly in "space" -- across the 2D image (y-axis, wavelength)
2. Features that change quickly in time, and slowly in "space"

The first case is treated by applying a rolling trimmed mean to each raster along the time dimension, resulting in an averaged array that is blurred in time, but adjusts quickly in "space" to treat that component.
The second case is treated by fitting each image with a low-order 2D polynomial, which results in an average array that is blurred in "space", but adjusts quickly in time to take care of that component.

### Mask out spectral lines
Both steps of the background image creation rely on masking out the spectral lines, which are then filled in to complete the background image.

#### Appoximate the line tilt
The spectral lines are tilted since geometric correction has not yet been applied, so estimate functions for lines that approximately follow the two Si IV lines.

In [ ]:
# Estimate line positions of Si IV 1394 & 1403

# upper point
max_upper_1394 = np.argmax(np.nanmean(average_1394_mean_image[-3:~0], axis=0))
max_upper_1403 = np.argmax(np.nanmean(average_1403_mean_image[-3:~0], axis=0))

# lower point
max_lower_1394 = np.argmax(np.nanmean(average_1394_mean_image[0:3], axis=0))
max_lower_1403 = np.argmax(np.nanmean(average_1403_mean_image[0:3], axis=0))

# slope of the line
slope_1394 = num_y / (max_upper_1394 - max_lower_1394)
slope_1403 = num_y / (max_upper_1403 - max_lower_1403)

# y intercept
y_intercept_1394 = (0 - max_lower_1394) * (num_y / (max_upper_1394 - max_lower_1394))
y_intercept_1403 = (0 - max_lower_1403) * (num_y / (max_upper_1403 - max_lower_1403))

# Generate 1394 grid
x_values_1394 = np.arange(0, num_x_1394)
y_values_1394 = np.arange(0, num_y)
x_grid_1394, y_grid_1394 = np.meshgrid(x_values_1394, y_values_1394)

# Generate 1403 grid
x_values_1403 = np.arange(0, num_x_1403)
y_values_1403 = np.arange(0, num_y)
x_grid_1403, y_grid_1403 = np.meshgrid(x_values_1403, y_values_1403)

# Invert the equation for a line, so we can add or subtract along the x-axis (approximately doppler velocity)
x_line_1394 = (y_grid_1394 - y_intercept_1394) / slope_1394
x_line_1403 = (y_grid_1403 - y_intercept_1403) / slope_1403


#### Define functions to convert doppler velocity to pixels
Very approximate -- images are not geometrically-corrected yet

In [ ]:
# Line center wavelength
lambda_1394 = 1393.757  # Angstrom
lambda_1403 = 1402.772  # Angstrom

# Dispersion
# delta_lambda = 0.05 # Angstrom/pix
# Dispersion is different for the August 2024 mosaic
delta_lambda = 0.025 # Angstrom/pix

# Find wavelength for a given doppler velocity
def doppler_lambda(lambda0, v):
    l = lambda0 * (1 - (v / const.c.to(u.km/u.s).value))
    return l

# Find pixel that corresponds most closely to doppler velocity of interest
def num_pts(lambda0, lambda_dop):
    n = abs(round((lambda0 - lambda_dop)/delta_lambda))
    return n

Find the pixels affected by the fiducials

In [ ]:
plt.figure(figsize=(8,25))
plt.imshow(average_1394_mean_image, vmax=2, vmin=-2, origin='lower')
plt.axhline(y=378, ls=':', lw=0.5, c='r')
plt.axhline(y=382, ls=':', lw=0.5, c='r')
plt.ylim((370,390))
plt.figure(figsize=(12,25))
plt.imshow(average_1403_mean_image, vmax=2, vmin=-2, origin='lower')
plt.axhline(y=380, ls=':', lw=0.5, c='r')
plt.axhline(y=385, ls=':', lw=0.5, c='r')
plt.ylim((370,390))

#### Create spectral line masks

In [ ]:
# For older mosaics, the fiducials are in the spectrograph images still
# Mask them out as well
# Actually... maybe leave it alone unless it proves to be a problem
# fid_mask_1394 = ((y_grid_1394 < 82) & (y_grid_1394 > 78))
# fid_mask_1394[(y_grid_1394 < 112) & (y_grid_1394 > 109)] = True
# fid_mask_1394[(y_grid_1394 < 382) & (y_grid_1394 > 378)] = True
#
# fid_mask_1403 = ((y_grid_1403 < 85) & (y_grid_1403 > 80))
# fid_mask_1403[(y_grid_1403 < 115) & (y_grid_1403 > 111)] = True
# fid_mask_1403[(y_grid_1403 < 385) & (y_grid_1403 > 380)] = True

# Mask off +/- 100 km/s around each side of the Si IV line

# Doppler velocity
# v = 60    # km/s
v = 90    # km/s

# Number of pixels corresponding to a given doppler velocity
n_1394_v = num_pts(lambda_1394, doppler_lambda(lambda_1394, v))
n_1403_v = num_pts(lambda_1403, doppler_lambda(lambda_1403, v))

# Pixel locations of other lines relative to the main Si IV 1394 line and their half width
delta_pix_line_2_1394 = -18
delta_pix_line_3_1394 = -24
delta_pix_line_4_1394 = -31
delta_pix_line_5_1394 = 46
half_width_2_1394 = 3
half_width_1394 = 2

# Multiply each by two for the August 2024 mosaic
delta_pix_line_2_1394 *= 2
delta_pix_line_3_1394 *= 2
delta_pix_line_4_1394 *= 2
delta_pix_line_5_1394 *= 2
half_width_2_1394 *= 2
half_width_1394 *= 2

# 1394 line mask ***** ADDITIONAL PIXELS MULTIPLIED BY TWO FOR AUGUST 2024 MOSAIC ****** REMOVE FOR NORMAL MOSAICS **********
line_mask_1394 = ((x_grid_1394 < x_line_1394 + (n_1394_v + (2*2))) &
                  (x_grid_1394 > x_line_1394 - (n_1394_v + (4*2)))) # Another few pix added to cover Nickel line on the blue side
line_mask_1394[(x_grid_1394 < x_line_1394 + half_width_2_1394 + delta_pix_line_2_1394 - 1) & # Reduce mask slightly on this side... seems to be room
               (x_grid_1394 > x_line_1394 - half_width_2_1394 + delta_pix_line_2_1394)] = True
line_mask_1394[(x_grid_1394 < x_line_1394 + half_width_1394 + delta_pix_line_3_1394) & 
               (x_grid_1394 > x_line_1394 - half_width_1394 + delta_pix_line_3_1394)] = True
line_mask_1394[(x_grid_1394 < x_line_1394 + half_width_1394 + delta_pix_line_4_1394) & 
               (x_grid_1394 > x_line_1394 - half_width_1394 + delta_pix_line_4_1394)] = True
line_mask_1394[(x_grid_1394 < x_line_1394 + half_width_1394 + delta_pix_line_5_1394) &
               (x_grid_1394 > x_line_1394 - half_width_1394 + delta_pix_line_5_1394)] = True

# Pixel locations of other lines relative to the main Si IV 1403 line and their half widths
delta_pix_line_2_1403 = -31
delta_pix_line_3_1403 = -59
delta_pix_line_4_1403 = -77
delta_pix_line_5_1403 = 40
delta_pix_line_6_1403 = 55
delta_pix_line_7_1403 = 64
half_width_2_1403 = 6
half_width_3_1403 = 4
half_width_4_1403 = 5
half_width_5_1403 = 5
half_width_6_1403 = 2
half_width_7_1403 = 2

# Multiply each by two for the August 2024 mosaic
delta_pix_line_2_1403 *= 2
delta_pix_line_3_1403 *= 2
delta_pix_line_4_1403 *= 2
delta_pix_line_5_1403 *= 2
delta_pix_line_6_1403 *= 2
delta_pix_line_7_1403 *= 2
half_width_2_1403 *= 2
half_width_3_1403 *= 2
half_width_4_1403 *= 2
half_width_5_1403 *= 2
half_width_6_1403 *= 2
half_width_7_1403 *= 2

# 1403 line mask
# line_mask_1403 = ((x_grid_1403 < x_line_1403 + (n_1403_v + 2)) &
#                   (x_grid_1403 > x_line_1403 - (n_1403_v + 4)))
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_2_1403 + delta_pix_line_2_1403 + 3) &
#                (x_grid_1403 > x_line_1403 - half_width_2_1403 + delta_pix_line_2_1403 - 6)] = True
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_3_1403 + delta_pix_line_3_1403) &
#                (x_grid_1403 > x_line_1403 - half_width_3_1403 + delta_pix_line_3_1403 - 1)] = True
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_4_1403 + delta_pix_line_4_1403) &
#                (x_grid_1403 > x_line_1403 - half_width_4_1403 + delta_pix_line_4_1403)] = True
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_5_1403 + delta_pix_line_5_1403) &
#                (x_grid_1403 > x_line_1403 - half_width_5_1403 + delta_pix_line_5_1403 - 2)] = True
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_6_1403 + delta_pix_line_6_1403) &
#                (x_grid_1403 > x_line_1403 - half_width_6_1403 + delta_pix_line_6_1403)] = True
# line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_7_1403 + delta_pix_line_7_1403) &
#                (x_grid_1403 > x_line_1403 - half_width_7_1403 + delta_pix_line_7_1403)] = True

 #***** ADDITIONAL PIXELS MULTIPLIED BY TWO FOR AUGUST 2024 MOSAIC ****** REMOVE FOR NORMAL MOSAICS **********
line_mask_1403 = ((x_grid_1403 < x_line_1403 + (n_1403_v + (2*2))) &
                  (x_grid_1403 > x_line_1403 - (n_1403_v + (4*2))))
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_2_1403 + delta_pix_line_2_1403 + (3*2)) &
               (x_grid_1403 > x_line_1403 - half_width_2_1403 + delta_pix_line_2_1403 - (6*2))] = True
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_3_1403 + delta_pix_line_3_1403) &
               (x_grid_1403 > x_line_1403 - half_width_3_1403 + delta_pix_line_3_1403 - (1*2))] = True
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_4_1403 + delta_pix_line_4_1403) &
               (x_grid_1403 > x_line_1403 - half_width_4_1403 + delta_pix_line_4_1403)] = True
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_5_1403 + delta_pix_line_5_1403) &
               (x_grid_1403 > x_line_1403 - half_width_5_1403 + delta_pix_line_5_1403 - (2*2))] = True
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_6_1403 + delta_pix_line_6_1403) &
               (x_grid_1403 > x_line_1403 - half_width_6_1403 + delta_pix_line_6_1403)] = True
line_mask_1403[(x_grid_1403 < x_line_1403 + half_width_7_1403 + delta_pix_line_7_1403) &
               (x_grid_1403 > x_line_1403 - half_width_7_1403 + delta_pix_line_7_1403)] = True

# Plots
plot_lines_sidebyside(
    # np.where(line_mask_1394 + fid_mask_1394, average_1394_mean_image, np.nan),
    # np.where(line_mask_1403 + fid_mask_1403, average_1403_mean_image, np.nan),
    np.where(line_mask_1394, average_1394_mean_image, np.nan),
    np.where(line_mask_1403, average_1403_mean_image, np.nan),
    'masked trimmed mean',
    size=(7,12),
    exp_max=4,
    exp_min=-3,
);

#### Apply masks to the trimmed mean data

In [ ]:
# Create masked array
average_1394_masked = average_1394.copy()
average_1403_masked = average_1403.copy()

# Mask line using NaNs
# average_1394_masked[..., line_mask_1394 + fid_mask_1394] = np.nan
# average_1403_masked[..., line_mask_1403 + fid_mask_1403] = np.nan
average_1394_masked[..., line_mask_1394] = np.nan
average_1403_masked[..., line_mask_1403] = np.nan

average_1394_mean_image_masked = np.nanmean(average_1394_masked, axis=0)
average_1403_mean_image_masked = np.nanmean(average_1403_masked, axis=0)

In [ ]:
plot_lines_sidebyside(
    average_1394_mean_image_masked,
    average_1403_mean_image_masked,
    'Masked average',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
    # exp_max=35,
    # exp_min=3
    );
 # .savefig('masked_average_mar_2024.png',dpi=300, transparent=True)

#### Delete arrays no longer in use

In [ ]:
del average_1394, average_1403

#### Smooth average images prior to filling
Makes more sense to do this step first since high and low points can easily bleed into the filled area while using Gauss-Seidel

In [ ]:
# kernel = astropy.convolution.Gaussian2DKernel(x_stddev=5,y_stddev=5,x_size=31,y_size=31).array
kernel = astropy.convolution.Gaussian2DKernel(x_stddev=10,y_stddev=10,x_size=61,y_size=61).array

plt.figure()
plt.imshow(kernel)

kernel = np.expand_dims(kernel, axis=0)

Smooth the upper and lower taps separately

In [ ]:
# August 2024 mosaic takes 2.5 hours...
average_1394_masked = np.concatenate([
    convolve(average_1394_masked[slice(None), sl_1394_up], kernel, boundary='extend', preserve_nan=True),
    convolve(average_1394_masked[slice(None), sl_1394_down], kernel, boundary='extend', preserve_nan=True)
], axis=1)

average_1403_masked = np.concatenate([
    convolve(average_1403_masked[slice(None), sl_1403_up], kernel, boundary='extend', preserve_nan=True),
    convolve(average_1403_masked[slice(None), sl_1403_down], kernel, boundary='extend', preserve_nan=True)
], axis=1)

In [ ]:
average_1394_masked_smooth_mean_image = np.nanmean(average_1394_masked, axis=0)
average_1403_masked_smooth_mean_image = np.nanmean(average_1403_masked, axis=0)

plot_lines_sidebyside(
    average_1394_masked_smooth_mean_image,
    average_1403_masked_smooth_mean_image,
    'Masked average',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
);

#### Divide average images by provided normalized background image
Remove as much of the complicated background as possible to make the filling step more accurate

Start by smoothing the normalized background with the same smoothing kernel used on the average set.

In [ ]:
# bg_1394_smooth = convolve(bg_1394, np.squeeze(kernel), boundary='extend', preserve_nan=True)

Divide the smoothed average set by the smoothed normalized background image. We only need the lower half of the 1394 line where the shadow is complicated.

In [ ]:
# average_1394_masked[slice(None), sl_1394_down] = average_1394_masked[slice(None), sl_1394_down] / bg_1394_smooth[sl_1394_down]

In [ ]:
# average_1394_masked_smooth_mean_image = np.nanmean(average_1394_masked, axis=0)
#
# plot_lines_sidebyside(
#     average_1394_masked_smooth_mean_image,
#     average_1403_masked_smooth_mean_image,
#     'Masked average',
#     size=(7,12),
#     exp_min=5,
#     exp_max=26
#     # percentile_min=.1,
#     # percentile_max=99.9,
# );

### Fill in masked areas

#### Try numba-accelerated Gauss-Seidel relaxation method
Doesn't quite seem to work near edges or in the notch

In [ ]:
# test_up = regridding.fill(average_1394_masked_smooth_mean_image[sl_1394_up], method='gauss_seidel', num_iterations=100, guess=np.nanmean(average_1394_masked_smooth_mean_image[sl_1394_up], axis=1, keepdims=True))
# test_down = regridding.fill(average_1394_masked_smooth_mean_image[sl_1394_down], method='gauss_seidel', num_iterations=100, guess=np.nanmean(average_1394_masked_smooth_mean_image[sl_1394_down], axis=1, keepdims=True))

In [ ]:
# test = np.concatenate([test_up, test_down], axis=0)

In [ ]:
# plt.figure()
# plt.imshow(test)
# plt.colorbar()

#### PyInterp Gauss-Seidel relaxation method
Seems to produce the best results so far, but takes about 14 hours to run both lines

In [ ]:
x_axis_1394 = pyinterp.Axis(np.arange(0, average_1394_mean_image[sl_1394_up].shape[1], 1.0))
y_axis_1394 = pyinterp.Axis(np.arange(0, average_1394_mean_image[sl_1394_up].shape[0], 1.0))

average_filled_1394 = np.empty(average_1394_masked.shape)

for i, image in enumerate(average_1394_masked):
    grid_1394_up = pyinterp.Grid2D(y_axis_1394, x_axis_1394, image[sl_1394_up])
    grid_1394_down = pyinterp.Grid2D(y_axis_1394, x_axis_1394, image[sl_1394_down])

    _, filled_1394_up = pyinterp.fill.gauss_seidel(grid_1394_up)
    _, filled_1394_down = pyinterp.fill.gauss_seidel(grid_1394_down)
    
    average_filled_1394[i][sl_1394_up] = filled_1394_up
    average_filled_1394[i][sl_1394_down] = filled_1394_down
    
del average_1394_masked

x_axis_1403 = pyinterp.Axis(np.arange(0, average_1403_mean_image[sl_1403_up].shape[1], 1.0))
y_axis_1403 = pyinterp.Axis(np.arange(0, average_1403_mean_image[sl_1403_up].shape[0], 1.0))

average_filled_1403 = np.empty(average_1403_masked.shape)

for i, image in enumerate(average_1403_masked):
    grid_1403_up = pyinterp.Grid2D(y_axis_1403, x_axis_1403, image[sl_1403_up])
    grid_1403_down = pyinterp.Grid2D(y_axis_1403, x_axis_1403, image[sl_1403_down])

    _, filled_1403_up = pyinterp.fill.gauss_seidel(grid_1403_up)
    _, filled_1403_down = pyinterp.fill.gauss_seidel(grid_1403_down)
    
    average_filled_1403[i][sl_1403_up] = filled_1403_up
    average_filled_1403[i][sl_1403_down] = filled_1403_down
    
del average_1403_masked

with open(path / 'trim_mean_painted_in_background_smooth_1394.pickle', 'wb') as fh:
    pickle.dump(average_filled_1394, fh)

with open(path / 'trim_mean_painted_in_background_smooth_1403.pickle', 'wb') as fh:
    pickle.dump(average_filled_1403, fh)

Reload data if necessary

In [ ]:
with open(path / 'trim_mean_painted_in_background_smooth_1394.pickle', 'rb') as f:
    average_filled_1394 = pickle.load(f)

with open(path / 'trim_mean_painted_in_background_smooth_1403.pickle', 'rb') as f:
    average_filled_1403 = pickle.load(f)

Correct the notched area of the 1394 line by multiplying by the smoothed provided FUV background image. Just undoing what we did before filling.

In [ ]:
# average_filled_1394[slice(None), sl_1394_down] = average_filled_1394[slice(None), sl_1394_down] * bg_1394_smooth[sl_1394_down]

Generate mean image of filled in background

In [ ]:
average_1394_filled_mean_image = np.nanmean(average_filled_1394, axis=0)
average_1403_filled_mean_image = np.nanmean(average_filled_1403, axis=0)

In [ ]:
plot_lines_sidebyside(
    average_1394_filled_mean_image,
    average_1403_filled_mean_image,
    'Gauss-Seidel mean results',
    size=(7,12),
    # exp_max=35,
    # exp_min=3
)

plt.figure()
plt.plot(np.mean(average_1394_filled_mean_image[0:264], axis=0))

plt.figure()
plt.plot(np.mean(average_1403_filled_mean_image[0:264], axis=0))
# .savefig('filled_average_mar_2024.png',dpi=300, transparent=True)

# plot_lines_sidebyside(
#     average_filled_1394[i],
#     average_filled_1403[i],
#     f'Gauss-Seidel results for image {i}',
#     size=(6,10),
# )

In [ ]:
bg_1394_step_1 = average_filled_1394
del average_filled_1394

bg_1403_step_1 = average_filled_1403
del average_filled_1403

In [ ]:
bg_1394_step_1_mean_image = np.mean(bg_1394_step_1, axis=0)
bg_1403_step_1_mean_image = np.mean(bg_1403_step_1, axis=0)

Plots

In [ ]:
plot_lines_sidebyside(
    sg_1394_dspk_mean_image,
    sg_1403_dspk_mean_image,
    'mean despiked image',
    size=(7,12),
    percentile_min=0,
    percentile_max=99,
);

In [ ]:
plot_lines_sidebyside(
    average_1394_mean_image_masked,
    average_1403_mean_image_masked,
    'Masked average',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
);

In [ ]:
plot_lines_sidebyside(
    average_1394_masked_smooth_mean_image,
    average_1403_masked_smooth_mean_image,
    'Masked average',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
);

In [ ]:
plot_lines_sidebyside(
    bg_1394_step_1_mean_image,
    bg_1403_step_1_mean_image,
    'step 1 background subtraction',
    # percentile_min=.1,
    # percentile_max=100,
    # exp_max=16,
    # exp_min=-4,
    size=(7,10),
);

In [ ]:
j = 64*5
jj = 64*6
plt.figure(figsize=(10,10))
plt.plot(np.sum(bg_1394_step_1_mean_image[j:jj], axis=0))
plt.plot(np.sum(average_1394_masked_smooth_mean_image[j:jj], axis=0))
plt.plot(np.sum(average_1394_mean_image_masked[j:jj], axis=0))
plt.plot(np.sum(sg_1394_dspk_mean_image[j:jj], axis=0))
plt.ylim((0, 100))

In [ ]:
j = 64*3
jj = 64*4
plt.figure(figsize=(15,10))
plt.plot(np.sum(bg_1403_step_1_mean_image[j:jj], axis=0))
plt.plot(np.sum(average_1403_masked_smooth_mean_image[j:jj], axis=0))
plt.plot(np.sum(average_1403_mean_image_masked[j:jj], axis=0))
plt.plot(np.sum(sg_1403_dspk_mean_image[j:jj], axis=0))
plt.ylim((20, 80))

In [ ]:
plot_lines_sidebyside(
    bg_1394_step_1_mean_image,
    bg_1403_step_1_mean_image,
    'step 1 background subtraction',
    # percentile_min=.1,
    # percentile_max=100,
    # exp_max=16,
    # exp_min=-4,
    size=(7,10),
);

In [ ]:
plot_lines_sidebyside(
    sg_1394_dspk_mean_image - bg_1394_step_1_mean_image,
    sg_1403_dspk_mean_image - bg_1403_step_1_mean_image,
    'step 1 background subtraction',
    # percentile_min=.1,
    # percentile_max=100,
    exp_max=20,
    exp_min=-5,
    size=(7,10),
);
#.savefig('step1_bg_sub_may_2025.png',dpi=300, transparent=True)

In [ ]:
plt.figure()
# plt.plot(np.nanmean(new_bg_1394_mean_image, axis=0))
plt.plot(np.nanmean(bg_1394_step_1_mean_image[400:], axis=0))
plt.plot(np.nanmean(bg_1394_step_1_mean_image[:128,7:], axis=0))
plt.plot(np.nanmean(sg_1394_dspk_mean_image[400:], axis=0), linewidth=0.5)
plt.plot(np.nanmean(sg_1394_dspk_mean_image[:128,7:], axis=0), linewidth=0.5)

In [ ]:
plt.figure()
# plt.plot(np.nanmean(bg_1403_step_1_mean_image, axis=0))
plt.plot(np.nanmean(bg_1403_step_1_mean_image[400:], axis=0))
plt.plot(np.nanmean(bg_1403_step_1_mean_image[:128,7:], axis=0))
plt.plot(np.nanmean(sg_1403_dspk_mean_image[400:], axis=0), linewidth=0.5)
plt.plot(np.nanmean(sg_1403_dspk_mean_image[:128,7:], axis=0), linewidth=0.5)

In [ ]:
# for file_idx, (file, dspk_1394, dspk_1403) in enumerate(zip(files, sg_1394_dspk, sg_1403_dspk)):
#     # Read in spectrograph image data
#     wcs, hdul = read_sg_image(file,'fuv2')
#     sg_image = hdul[0].data
#
#     sg_image[sl_1394] = dspk_1394
#     sg_image[sl_1403] = dspk_1403
#
#     hdul[0].data = sg_image
#     new_file = file.parent.parent / 'level_12' / file.name
#     hdul.writeto(new_file, overwrite=True)

In [ ]:
mean_step_1_bg_sub_1394 = sg_1394_dspk_mean_image - bg_1394_step_1_mean_image
plt.figure()
plt.plot(np.nanmean(mean_step_1_bg_sub_1394[400:], axis=0))
plt.plot(np.nanmean(mean_step_1_bg_sub_1394[:128,7:], axis=0))
plt.axhline(y=0, color='r', ls='--', lw=0.5)

In [ ]:
mean_bg_sub_1394 = sg_1394_dspk_mean_image - bg_1394_step_1_mean_image
plt.figure()
plt.plot(np.nanmean(mean_bg_sub_1394[400:], axis=0))
plt.plot(np.nanmean(mean_bg_sub_1394[:128,7:], axis=0))
plt.axhline(y=0)

In [ ]:
i = 996

plot_lines_sidebyside(
    bg_1394_step_1[i],
    bg_1403_step_1[i],
    f'new step 1 background (image {i})',
    # percentile_min=.1,
    # percentile_max=100,
    # exp_max=50,
    # exp_min=-10,
    size=(7,12),
)

plot_lines_sidebyside(
    sg_1394_dspk[i],
    sg_1403_dspk[i],
    f'original (image {i})',
    # percentile_min=.1,
    # percentile_max=100,
    # exp_max=50,
    # exp_min=-10,
    size=(7,12),
)

plot_lines_sidebyside(
    sg_1394_dspk[i] - bg_1394_step_1[i],
    sg_1403_dspk[i] - bg_1403_step_1[i],
    f'original - step 1 background (image {i})',
    # percentile_min=.1,
    # percentile_max=100,
    # exp_max=80,
    # exp_min=-10,
    size=(7,12),
);

Plots

In [ ]:
plot_lines_sidebyside(
    average_1394_mean_image - bg_1394_step_1_mean_image,
    average_1403_mean_image - bg_1403_step_1_mean_image,
    'trimmed mean - new background',
    size=(6,10),
)

plt.figure(figsize=(8,5))
plt.plot(np.sum(sg_1394_dspk_mean_image, axis=0), label='original')
plt.plot(np.sum(sg_1394_dspk_mean_image - bg_1394_step_1_mean_image, axis=0), label='original - new background')
plt.legend(loc='upper left')
plt.axhline(y=0, color='r', ls='--', lw=0.5)


plt.figure(figsize=(8,5))
plt.plot(np.sum(sg_1403_dspk_mean_image, axis=0), label='original')
plt.plot(np.sum(sg_1403_dspk_mean_image - bg_1403_step_1_mean_image, axis=0), label='original - new background')
plt.legend(loc='upper left')
plt.axhline(y=0, color='r', ls='--', lw=0.5)

#### Subtract off averaged background

In [ ]:
sg_1394_dspk -= bg_1394_step_1
sg_1403_dspk -= bg_1403_step_1

## Step 2: Low-order, 2D polynomial fit of remaining background

In [ ]:
del nan_mask_1394, nan_mask_1403

In [ ]:
del bg_1394_step_1, bg_1403_step_1

#### Fit remaining background in 1394 Å images using a low-order 2D polynomial

In [ ]:
p_init = models.Polynomial2D(degree=3)
fit_p = fitting.LevMarLSQFitter()

bg_step_2_1394 = np.empty(sg_1394_dspk.shape)

for i, img in enumerate(sg_1394_dspk):
    img = img.copy()

    # Mask main spectral lines
    img[line_mask_1394] = np.nan

    # Find the upper and lower quantiles of the remaining pixels in the image and set to NaNs outside of those bounds
    q_low, q_high = np.nanquantile(img, [0.1, 0.9])
    img[img>q_high] = np.nan
    img[img<q_low] = np.nan

    # Final NaN mask
    nan_mask = np.isfinite(img).astype(bool)

    if np.all(~nan_mask):   # If image is just NaNs, skip to the next image in the loop
        continue

    # Separate image and mask into upper and lower halves since the tap is a hard cutoff in the middle
    img_up = img[sl_1394_up]
    img_down = img[sl_1394_down]

    nan_up = nan_mask[sl_1394_up]
    nan_down = nan_mask[sl_1394_down]

    # Define x and y grid points
    y, x = np.mgrid[:num_y_split, :num_x_1394]

    # Fit upper half of 1394 image with 2D polynomial
    if not np.all(~nan_up):   # If all NaNs, skip this quadrant of the image

        bg_fit_up = fit_p(p_init,
                               x[nan_up],
                               y[nan_up],
                               img_up[nan_up])

        bg_fit_up = bg_fit_up(x, y)

        # Resulting upper half of the fit
        bg_step_2_1394[i][sl_1394_up] = bg_fit_up

    # Fit lower half of 1394 image with 2D polynomial
    if not np.all(~nan_down):   # If all NaNs, skip this quadrant of the image

        bg_fit_down = fit_p(p_init,
                               x[nan_down],
                               y[nan_down],
                               img_down[nan_down])

        bg_fit_down = bg_fit_down(x, y)

        # Resulting lower half of the fit
        bg_step_2_1394[i][sl_1394_down] = bg_fit_down

# Save data
with open(path / 'polynomial_fit_bgsub_step_2_1394.pickle', 'wb') as fh:
    pickle.dump(bg_step_2_1394, fh)

#### Fit remaining background in 1403 Å images using a low-order 2D polynomial

In [ ]:
bg_step_2_1403 = np.empty(sg_1403_dspk.shape)

for i, img in enumerate(sg_1403_dspk):
    img = img.copy()

    # Mask main spectral lines
    img[line_mask_1403] = np.nan

    # Find the upper and lower quantiles of the remaining pixels in the image and set to NaNs outside of those bounds
    q_low, q_high = np.nanquantile(img, [0.1, 0.9])
    img[img>q_high] = np.nan
    img[img<q_low] = np.nan

    # Final NaN mask
    nan_mask = np.isfinite(img).astype(bool)

    if np.all(~nan_mask):   # If image is just NaNs, skip to the next image in the loop
        continue

    # Separate image and mask into upper and lower halves since the tap is a hard cutoff in the middle
    img_up = img[sl_1403_up]
    img_down = img[sl_1403_down]

    nan_up = nan_mask[sl_1403_up]
    nan_down = nan_mask[sl_1403_down]

    # Define x and y grid points
    y, x = np.mgrid[:num_y_split, :num_x_1403]

    # Fit upper half of 1394 image with 2D polynomial
    if not np.all(~nan_up):   # If all NaNs, skip this quadrant of the image

        bg_fit_up = fit_p(p_init,
                               x[nan_up],
                               y[nan_up],
                               img_up[nan_up])

        bg_fit_up = bg_fit_up(x, y)

        # Resulting upper half of the fit
        bg_step_2_1403[i][sl_1403_up] = bg_fit_up

    # Fit lower half of 1394 image with 2D polynomial
    if not np.all(~nan_down):   # If all NaNs, skip this quadrant of the image

        bg_fit_down = fit_p(p_init,
                               x[nan_down],
                               y[nan_down],
                               img_down[nan_down])

        bg_fit_down = bg_fit_down(x, y)

        # Resulting lower half of the fit
        bg_step_2_1403[i][sl_1403_down] = bg_fit_down

# Save data
with open(path / 'polynomial_fit_bgsub_step_2_1403.pickle', 'wb') as fh:
    pickle.dump(bg_step_2_1403, fh)

Reload data if necessary

In [ ]:
with open(path / 'polynomial_fit_bgsub_step_2_1394.pickle', 'rb') as f:
    bg_step_2_1394 = pickle.load(f)

with open(path / 'polynomial_fit_bgsub_step_2_1403.pickle', 'rb') as f:
    bg_step_2_1403 = pickle.load(f)

In [ ]:
bg_1394_step_2_mean_image = np.nanmean(bg_step_2_1394, axis=0)
bg_1403_step_2_mean_image = np.nanmean(bg_step_2_1403, axis=0)

In [ ]:
plot_lines_sidebyside(
    bg_1394_step_2_mean_image,
    bg_1403_step_2_mean_image,
    'step 2 background mean image',
    size=(6,10),
);

In [ ]:
# Average each background fit image to get a single number
bg_step_2_1394_avg_value_each_image = np.nanmean(bg_step_2_1394, axis=(1,2))
bg_step_2_1403_avg_value_each_image = np.nanmean(bg_step_2_1403, axis=(1,2))

In [ ]:
# Identify the image index of the brightest ones
i = 0
j = ~0
# i = 6000
# j = 6500
plt.figure()
plt.plot(bg_step_2_1394_avg_value_each_image[i:j])
plt.plot(bg_step_2_1403_avg_value_each_image[i:j])
plt.axhline(0.4, color='k', lw=0.5)
plt.axhline(-0.325, color='k', lw=0.5)

test_img_index = np.argmax(bg_step_2_1394_avg_value_each_image[i:j]) + i
# test_img_index = np.argmin(bg_step_2_1394_avg_value_each_image[i:j]) + i
print(test_img_index)
print('1394: ',bg_step_2_1394_avg_value_each_image[test_img_index])
print('1403: ',bg_step_2_1403_avg_value_each_image[test_img_index])

In [ ]:
test_img = sg_1394_dspk[test_img_index].copy()

test_img[line_mask_1394] = np.nan

q_low, q_high = np.nanquantile(test_img, [0.1, 0.9])
test_img[test_img>q_high] = np.nan
test_img[test_img<q_low] = np.nan

plt.figure(figsize=(3,10))
plt.imshow(test_img, origin='lower')
plt.colorbar()

In [ ]:
# Verify that the fit would subtract off too much background, check a few images

plot_lines_sidebyside(
    bg_step_2_1394[test_img_index],
    bg_step_2_1403[test_img_index],
    'step 2 background test image',
    size=(6,10),
)

plot_lines_sidebyside(
    sg_1394_dspk[test_img_index],
    sg_1403_dspk[test_img_index],
    'original test image',
    size=(6,10),
    exp_max=20,
    exp_min=-3
)

plot_lines_sidebyside(
    np.where(~line_mask_1394, sg_1394_dspk[test_img_index], np.nan),
    np.where(~line_mask_1403, sg_1403_dspk[test_img_index], np.nan),
    'original test image',
    size=(6,10),
    exp_max=20,
    exp_min=-3
)

plot_lines_sidebyside(
    sg_1394_dspk[test_img_index] - bg_step_2_1394[test_img_index],
    sg_1403_dspk[test_img_index] - bg_step_2_1403[test_img_index],
    'original test image',
    size=(6,10),
    exp_max=20,
    exp_min=-3
)

plt.figure()
plt.plot(np.nanmean(sg_1394_dspk[test_img_index], axis=0))
plt.plot(np.nanmean(sg_1394_dspk[test_img_index] - bg_step_2_1394[test_img_index], axis=0))
plt.axhline(0, color='r', lw=0.5, ls=':')

plt.figure()
plt.plot(np.nanmean(sg_1403_dspk[test_img_index], axis=0))
plt.plot(np.nanmean(sg_1403_dspk[test_img_index] - bg_step_2_1403[test_img_index], axis=0))
plt.axhline(0, color='r', lw=0.5, ls=':')

#### Demonstrate how the "stripes" have disappeared

In [ ]:
plt.figure()
plt.imshow(sg_1394_dspk[0:64,:,3].T, aspect=1/6, vmax=8)
plt.title('single raster, within notch, step 1 bg subtraction only');

In [ ]:
plt.figure()
plt.imshow(sg_1394_dspk[0:64,:,3].T, aspect=1/6, vmax=8)
plt.title('single raster, within notch, step 1 bg subtraction only');

In [ ]:
plt.figure()
plt.imshow(sg_1394_dspk[0:64,:,3].T - bg_step_2_1394[0:64,:,3].T, aspect=1/6, vmax=8)
plt.title('single raster, within notch, step 1 & 2 bg subtraction');

In [ ]:
plt.figure()
plt.imshow(sg_1394_dspk[0:64,:,3].T - bg_step_2_1394[0:64,:,3].T, aspect=1/6, vmax=8)
plt.title('single raster, within notch, step 1 & 2 bg subtraction');

In [ ]:
plt.figure()
plt.imshow(bg_step_2_1394[0:64,:,3].T, aspect=1/6)
plt.colorbar()
plt.title('single raster, within notch, step 2 bg fit');

In [ ]:
mean_total_bg_sub_1394_1 = sg_1394_dspk_mean_image - bg_1394_step_1_mean_image
plt.figure()
plt.plot(np.nanmean(mean_total_bg_sub_1394_1[400:], axis=0))
plt.plot(np.nanmean(mean_total_bg_sub_1394_1[:128,7:], axis=0))
plt.axhline(y=0)

In [ ]:
mean_total_bg_sub_1394_2 = sg_1394_dspk_mean_image - bg_1394_step_1_mean_image - bg_1394_step_2_mean_image
plt.figure(figsize=(10,3))
plt.plot(np.nanmean(mean_total_bg_sub_1394_2[400:], axis=0))
plt.plot(np.nanmean(mean_total_bg_sub_1394_1[400:], axis=0), lw=0.5)
plt.plot(np.nanmean(mean_total_bg_sub_1394_2[:128,7:], axis=0))
plt.plot(np.nanmean(mean_total_bg_sub_1394_1[:128,7:], axis=0), lw=0.5)
plt.axhline(y=0)
plt.ylim((-0.2,.75))

In [ ]:
plt.figure(figsize=(7,7))
plt.plot(np.nanmean(mean_total_bg_sub_1394_1, axis=0))
plt.plot(np.nanmean(mean_total_bg_sub_1394_2, axis=0))
plt.axhline(y=0, ls='--')
# plt.ylim((-0.2,2))

#### Subtract off polynomial fits of remaining background

In [ ]:
sg_1394_dspk -= bg_step_2_1394
sg_1403_dspk -= bg_step_2_1403

#### Mean final background-subtracted images

In [ ]:
sg_1394_dspk_mean_image_bg_sub_final = np.nanmean(sg_1394_dspk, axis=0)
sg_1403_dspk_mean_image_bg_sub_final = np.nanmean(sg_1403_dspk, axis=0)

In [ ]:
plot_lines_sidebyside(
    sg_1394_dspk_mean_image_bg_sub_final,
    sg_1403_dspk_mean_image_bg_sub_final,
    'final background-subtracted mean image',
    size=(6,10),
    exp_min=-5,
    exp_max=20
);

#### Replace original data with our background-subracted data and save

In [ ]:
for i, (file, img_1394, img_1403) in enumerate(zip(files, sg_1394_dspk, sg_1403_dspk)):
    wcs, hdul, _ = read_sg_image(file,'fuv2')
    sg_image = hdul[0].data
    sg_image[sl_1394] = img_1394
    sg_image[sl_1403] = img_1403
    hdul[0].data = sg_image
    new_file = file.parent.parent / 'level_12' / file.name
    hdul.writeto(new_file, overwrite=True)